In [0]:
%sql
USE CATALOG project_lakehouse_telemetria_ele;
USE SCHEMA silver;

In [0]:
%sql
-- 1. Maiores saltos de temperatura (LAG)

WITH d AS (
  SELECT device_id, event_ts, temperatura_c,
         temperatura_c - LAG(temperatura_c) OVER (PARTITION BY device_id ORDER BY event_ts, row_hash) AS delta_temp
  FROM telemetria_clean WHERE NOT temp_pico
  
)
SELECT * FROM d ORDER BY ABS(delta_temp) DESC LIMIT 10;


In [0]:
%sql
-- 2. Média móvel de 10 leituras (ajuste o device_id e a data para valores que existam nos seus dados)
SELECT device_id, event_ts, temperatura_c,
       ROUND(AVG(temperatura_c) OVER (
         PARTITION BY device_id ORDER BY event_ts, row_hash
         ROWS BETWEEN 9 PRECEDING AND CURRENT ROW), 2) AS media_movel_10
FROM project_lakehouse_telemetria_ele.silver.telemetria_clean
WHERE device_id = 'TRF-000'
  AND data_evento = (SELECT MIN(data_evento) FROM project_lakehouse_telemetria_ele.silver.telemetria_clean)
ORDER BY event_ts, row_hash;

In [0]:
%sql
-- 3. Maiores intervalos sem leitura
WITH g AS (
  SELECT device_id, event_ts,
         unix_timestamp(event_ts) - unix_timestamp(
           LAG(event_ts) OVER (PARTITION BY device_id ORDER BY event_ts, row_hash)) AS gap_seg
  FROM project_lakehouse_telemetria_ele.silver.telemetria_clean
)
SELECT * FROM g ORDER BY gap_seg DESC LIMIT 10;

In [0]:
%sql
-- 4. Ranking de transformadores por picos (semente da Fase 3)
SELECT device_id, COUNT_IF(temp_pico) AS picos,
       RANK() OVER (ORDER BY COUNT_IF(temp_pico) DESC) AS ranking
FROM telemetria_clean
GROUP BY device_id
ORDER BY ranking
LIMIT 10;